<a href="https://colab.research.google.com/github/dr-bankert-augustana/PHYS_200/blob/main/Lessons/Module_5/Lesson_5_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="Notebook-Start"></a>

---

<font size = 7> <b> Lesson 5.4 (Bungee Jump) </b> </font>

---

<font size = 5> <b> Notebook Index </b> </font>

1. [Learning Outcomes](#Learning-Outcomes)

2. [Introduction](#Introduction)

3. [Adding Rope Mass](#RopeMass)
$
\newcommand{\lsum}{\displaystyle \sum\limits_{i=1}^{N}}
\newcommand{\parens}[1]{\left(#1\right)}
\newcommand{\dsfrac}[2]{\displaystyle\frac{#1}{#2}}
\newcommand{\dpfrac}[2]{\displaystyle\parens{\frac{#1}{#2}}}
\newcommand{\parderiv}[2]{\dsfrac{\partial #1}{\partial #2}}
\newcommand{\spc}{\hspace{0.1 pc}}
\newcommand{\ra}{\Rightarrow}
\newcommand{\of}[1]{{\scriptsize (#1)}}
\newcommand{\rule}{\Huge \hspace{-0.2 pc} \displaystyle\frac{\hspace{20 pc}}{\hspace{20 pc}}}
\newcommand{\mps}{\spc \frac{\textrm{m}}{\textrm{s}}}
\newcommand{\mpss}{\spc \frac{\textrm{m}}{\textrm{s}^2}}
$

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 1. Learning Outcomes </b> </font>

---

<font size = 5> <b> Learning Outcomes: </b> </font>

By the end of this lesson, students will be able to:

  <br>

  1. <b>Explain</b>

[Return to Top](#Notebook-Start)

<a name="Introduction"></a>

---

#<font size = 6> <b> 2. Introduction </b> </font>

---

##<font size = 5> <b> 2.1 Adding New Forces</b> </font>

In simulating the falling object problem using differential equations, we have made it possible to handle much more complex acceleration problems relatively easily.  

<br>

Why is it easier now? Addition! The Second Law of Motion connects a sum of forces to the second derivative of position (acceleration, that is):

<br>

$$\frac{d^2x}{dt^2} = \frac{\Sigma F}{m}$$

<br>

That sigma $\Sigma$ indicates addition, and this is not hard to handle in a simulation.  We can add whatever forces we'd like to our simulation--even if they change with time or position--and then use the differential equation to determine position and velocity for our object.

<br>

We'll test this out by setting up an experiment that allows us to just touch the ground at zero speed after plummeting down a couple hundred feet.

<br>

##<font size = 5> <b> 2.2 Modeling the Bungee Jump</b> </font>

Let's say that you want to do a daring bungee jump stunt, in which you jump off of an 80 meter tall crane, plummet to the Earth and reach the ground slow enough to just tap your feet on the pavement before shooting back up into the air.

<br>

We'll start with the following modeling assumptions:

-   Initially the bungee cord hangs from a crane with the attachment point 80 m above the ground.

-   Until the cord is fully extended, it applies no force to the jumper.  

-   After the cord is fully extended, it obeys Hooke's Law; that is, it applies a force to the jumper proportional to the extension of the cord beyond its resting length: $\vec{F} = -k \spc \Delta \vec{x}$

-   The mass of the jumper is 75 kg.

-   The bungee-cord has a spring constant of 40 N/m.

-   The jumper is subject to drag force so that their terminal velocity is 70 m/s; this is roughly the terminal velocity of a skydiver who is oriented vertically.

<br>

Our objective is to choose the length of the cord, `L`, so that the jumper falls all the way to the ground, but no farther!

### <b> Import Libraries and Define Useful Functions </b>

In [ ]:
##=============================================================================================##
## Import Libraries:                                                                           ##
##=============================================================================================##

import time as tm
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.optimize as spo
import matplotlib.pyplot as plt

from IPython.display import display_html

In [ ]:
#@title This cell defines the functions: display_dataframes

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  dataframe_list - List of DataFrames to be displayed                              ##
##            title_list     - List of titles for the displayed DataFrames                     ##
##            n_items        - Number of items to display (optional, default = 5)              ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(dataframe_list, title_list, n_items = 5, tail = False):

  ##===========================================================================================##
  ## Create a String for Housing the Commands to Be Sent to the display_html() Function:       ##
  ##===========================================================================================##

  html_str = ""

  ##===========================================================================================##
  ## Loop Over the Elements in the item_list and title_list:                                   ##
  ##===========================================================================================##

  for df, title in zip(dataframe_list, title_list):

    # Convert the current dataframe info to html:

    if (tail == True):

      html_df = pd.DataFrame(df).tail(n_items).to_html()

    else:

      html_df = pd.DataFrame(df).head(n_items).to_html()

    ##=========================================================================================##
    ## Wrap title and DataFrames in a Styled HTML <div>:                                       ##
    ##=========================================================================================##

    html_str += "<div style='display: inline-block; margin-right: 20px; vertical-align: top;'>"

    html_str += "<h3 style='text-align: center;'>" + str(title) + "</h3><hr>" + str(html_df)

    html_str += "</div>"

  ##===========================================================================================##
  ## Send the HTML String to the display_html() Function:                                      ##
  ##===========================================================================================##

  display_html(html_str, raw = True)

##=============================================================================================##
## Create a Function to Make Plots:                                                            ##
##=============================================================================================##

def plot_data(data_list, x_list = [], y_list = [], color_list = [], title_list = [],
              x_label_list = [], y_label_list = [], aspect_list = [], legend_list = [],
              overplot = False):

  ##===========================================================================================##
  ## If overplot is True, create an overplot of the graphs:                                    ##
  ##===========================================================================================##

  if (overplot == True):

    # Create the graph:

    x = x_list[0]
    y = y_list[0]

    x_size = 10
    y_size = 10 * aspect_list[0]

    ax = plt.figure(figsize = (x_size, y_size)).add_subplot(111)

    # Set the graph title:

    ax.set_title(title_list[0], fontsize = 14)

    # Set the Graph x-axis label:

    ax.set_xlabel(x_label_list[0], fontsize = 12)

    # Set the Graph y-axis label:

    ax.set_ylabel(y_label_list[0], fontsize = 12)

    # Add grid and the legend:

    ax.grid(True, linestyle = '--', alpha = 0.5)

    # Overplot the remaining data:

    for data, color, label in zip(data_list, color_list, legend_list):

      ax.plot(data[x], data[y], color = color, label = label)

    # Display the plots:

    plt.legend()

    plt.show()

  ##===========================================================================================##
  ## If overplot is False, create a series of graphs:                                          ##
  ##===========================================================================================##

  else:

    # Get the settings for each graph to be made:

    for X, Y, color, title, x_label, y_label, aspect in \
      zip(x_list, y_list, color_list, title_list, x_label_list, y_label_list, aspect_list):

      # Create the graph:

      x_size = 10
      y_size = 10 * aspect

      ax = data_list[0].plot(
        kind    = 'line',
        y       = [Y],
        x       = str(X),
        color   = [color],
        figsize = (x_size, y_size)
      )

      # Set the graph title:

      ax.set_title(title, fontsize = 14)

      # Set the Graph x-axis label:

      ax.set_xlabel(x_label, fontsize = 12)

      # Set the Graph y-axis label:

      ax.set_ylabel(y_label, fontsize = 12)

      # Add grid and suppress the legend:

      ax.grid(True, linestyle = '--', alpha = 0.5)

      legend = ax.legend(title = "Model")

      legend.remove()

    # Display the plots:

    plt.show()

##<font size = 5> <b> 2.3 Parameters and the system</b> </font>

We'll set up our state and parameters in the same way we did for the falling object problem, including calculating the coefficient of drag. We also make an initial guess at our independent variable, the length of the rope $L$.

<br>

The coefficient of drag is 0.83, which is a reasonable number given the situation.

In [ ]:
##=============================================================================================##
## Set the Simulation Parameters:                                                              ##
##=============================================================================================##

# Local acceleration due to gravity (in m/s^2):

g = 9.8

# Terminal velocity (in m/s):

v_term = 70

# Air mass density (in kg/m^3):

rho = 1.2

# Object mass (in kg):

mass = 75

# Object's cross-sectional area (in m^2):

area = 0.3

# Calculate the object's drag coeffcient:

C_d = (2.0 * mass * g) / (rho * v_term**2 * area)

print("The object's drag coefficient is: " + str(np.round(C_d, 3)))

# Bungee cord's attachment location (in m):

y_attach = 80

# Bungee cord's unstretched length (in m):

length = 25.0

# Bungee cord's spring constant (in N / m):

k = 40.0

# Time step size (in seconds):

dt = 0.01

# Simulation end time (in seconds):

t_end = 30

##=============================================================================================##
## Store the parameters in a Pandas Series:                                                    ##
##=============================================================================================##

parameters_bungee = pd.Series({
    "g":  g,
    "Cd": C_d,
    "rho": rho,
    "mass": mass,
    "area": area,
    "y_attach": y_attach,
    "length": length,
    "k": k,
    "dt": dt,
    "t_end": t_end
})

# Rename the columns:

parameters_bungee.name = 'Value'

parameters_bungee.index.name = "Parameter"

##=============================================================================================##
## Create the State Object                                                                     ##
##=============================================================================================##

# Initial Height:

y_initial = 80.0 # meters

# Initial Velocity:

v_initial = 0 # meters per second

# Initial Time:

t_initial = 0 # seconds

##=============================================================================================##
## Store the initial state in a Pandas Series:                                                 ##
##=============================================================================================##

initial_state = pd.Series({
    "Position" : y_initial,
    "Velocity" : v_initial,
    "Time"     : t_initial
})

initial_state.name = 'Value'

initial_state.index.name = "State"

##=============================================================================================##
## Display the Parameters and Initial State:                                                   ##
##=============================================================================================##

display_dataframes([parameters_bungee, initial_state],
                   ["Bungee Jump Parameters", "Initial State"], n_items = 15)

##<font size = 5> <b> 2.4 Computing Forces </b> </font>

Now we need to calculate the forces on the jumper.  In addition to the local force due to gravity, we'll add in the spring_force, which computes the force of the cord on the jumper.  If the spring is not extended, the spring force is set to 0; if it is extended, the force is calculated using Hooke's Law:

<br>

$$F_{spring} = k \Delta y$$

<br>

where $\Delta y$ is the distance that the spring (or rope) is stretched.

<br>

The spring force is 0 until the cord is fully extended.  Remember that the total height is 80 m, so as long as the jumper's position is $y=55~m$ or above, the cord is not extended.

<br>

When it is extended 1 m, the spring force is 40 N in the positive (upward) direction.

### <b>The Acceleration Functions</b>

We're going to create two different acceleration functions, one with drag and one without, so that we can see the effect that air drag has on our system.

In each case, we sum the included forces and determine the overall acceleration using the Second Law and addition:

In [ ]:
##=============================================================================================##
## Define the Acceleration Function For the Falling Object Simulation:                         ##
##=============================================================================================##

def acceleration_bungee_1d(state, parameters):

  ##===========================================================================================##
  ## Unpack the state and parameters:                                                          ##
  ##===========================================================================================##

  # Get the local acceleration of gravity:

  g = parameters["g"]

  # Get the bungee cord's unstretched length:

  L = parameters["length"]

  # Get the bungee cord's spring constant:

  k = parameters["k"]

  # Get the object's mass:

  mass = parameters["mass"]

  # Get the object's initial position:

  y_attach = parameters["y_attach"]

  # Get the object's current position:

  y = state["Position"]

  ##===========================================================================================##
  ## Calculate the Net Force on the Object:                                                    ##
  ##===========================================================================================##

  # Calculate the force due to local gravitational acceleration:

  f_grav = - mass * g

  # Calculate the force due to the bungee cord's spring-like nature:

  distance_to_attach = np.abs(y_attach - y)

  extension = distance_to_attach - L

  f_bungee = np.sign(y_attach - y) * k * np.maximum(extension, 0)

  # Calculate the force due to air drag (force is always in the opposite direction of v):

  #f_drag = - np.sign(v) * rho * v**2 * C_d * area / 2.0

  # Calculate the Net force:

  f_net = f_grav + f_bungee

  ##===========================================================================================##
  ## Return the Acceleration:                                                                  ##
  ##===========================================================================================##

  return f_net / mass

Now we add in the air drag force as a function of velocity. Notice how we use the NumPy function `sign()` to make sure that the force is always in the direction opposite to the direction of motion. `sign()` returns a value of either 1.0 or -1.0 (depending on the value used as an argument).

In [ ]:
##=============================================================================================##
## Define the Acceleration Function For the Falling Object Simulation:                         ##
##=============================================================================================##

def acceleration_bungee_drag_1d(state, parameters):

  ##===========================================================================================##
  ## Unpack the state and parameters:                                                          ##
  ##===========================================================================================##

  # Get the local acceleration of gravity:

  g = parameters["g"]

  # Get the bungee cord's unstretched length:

  L = parameters["length"]

  # Get the bungee cord's spring constant:

  k = parameters["k"]

  # Get the object's mass:

  mass = parameters["mass"]

  # Get the object's initial position:

  y_attach = parameters["y_attach"]

  # Get the object's current position:

  y = state["Position"]

  # Get the object's current velocity

  v = state["Velocity"]

  # Get the air density:

  rho = parameters["rho"]

  # Get the object's area:

  area = parameters["area"]

  # Get the object's drag coefficient:

  C_d = parameters["Cd"]

  ##===========================================================================================##
  ## Calculate the Net Force on the Object:                                                    ##
  ##===========================================================================================##

  # Calculate the force due to local gravitational acceleration:

  f_grav = - mass * g

  # Calculate the force due to air drag (force is always in the opposite direction of v):

  f_drag = - np.sign(v) * rho * v**2 * C_d * area / 2.0

  # Calculate the force due to the bungee cord's spring-like nature:

  distance_to_attach = np.abs(y_attach - y)

  extension = distance_to_attach - L

  f_bungee = np.sign(y_attach - y) * k * np.maximum(extension, 0)

  # Calculate the Net force:

  f_net = f_grav + f_bungee + f_drag

  ##===========================================================================================##
  ## Return the Acceleration:                                                                  ##
  ##===========================================================================================##

  return f_net / mass

### <b>Include Our Differential Equation Solver Algorithms</b>

In [ ]:
#@title This cell defines the Euler, Euler-Cromer, Euler-Richardson, and Runge-Kutta 4th Order DEQ Solver Algorithms.

##=============================================================================================##
## Define the 1D Euler Algorithm Change Function:                                              ##
##=============================================================================================##

def cf_euler_1d(state, parameters, acceleration):

  ##===========================================================================================##
  ## Prepare the Algorithm Values:                                                             ##
  ##===========================================================================================##

  # Create a local copy of the state object:

  state_local = state.copy()

  # Get the time step size:

  dt = parameters['dt']

  # Get the current position:

  x = state_local['Position']

  # Get the current velocity:

  v = state_local['Velocity']

  # Get the current acceleration:

  a = acceleration(state_local, parameters)

  # Update the position:

  x += v * dt

  # Update the velocity:

  v += a * dt

  ##===========================================================================================##
  ## Update the State:                                                                         ##
  ##===========================================================================================##

  # Update the local state position:

  state_local['Position'] = x

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  ##===========================================================================================##
  ## Return the Updated State:                                                                 ##
  ##===========================================================================================##

  return state_local

##=============================================================================================##
## Define the 1D Euler-Cromer Algorithm Function:                                              ##
##=============================================================================================##

def cf_euler_cromer_1d(state, parameters, acceleration):

  ##===========================================================================================##
  ## Prepare the Algorithm Values:                                                             ##
  ##===========================================================================================##

  # Create a local copy of the state object:

  state_local = state.copy()

  # Get the time step size:

  dt = parameters['dt']

  # Get the current position:

  x = state_local['Position']

  # Get the current velocity:

  v = state_local['Velocity']

  # Get the current acceleration:

  a = acceleration(state_local, parameters)

  # Update the velocity:

  v += a * dt

  # Update the position:

  x += v * dt

  ##===========================================================================================##
  ## Update the State:                                                                         ##
  ##===========================================================================================##

  # Update the local state position:

  state_local['Position'] = x

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  ##===========================================================================================##
  ## Return the Updated State:                                                                 ##
  ##===========================================================================================##

  return state_local

##=============================================================================================##
## Define the 1D Euler Algorithm Function:                                                     ##
##=============================================================================================##

def cf_euler_richardson_1d(state, parameters, acceleration):

  ##===========================================================================================##
  ## Prepare the Algorithm Values:                                                             ##
  ##===========================================================================================##

  # Create a local copy of the state object:

  state_local = state.copy()

  # Get the time step size:

  dt = parameters['dt']

  # Get the current position:

  x = state_local['Position']

  # Get the current velocity:

  v = state_local['Velocity']

  # Get the current acceleration:

  a = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Get the Midpoint Values:                                                                  ##
  ##===========================================================================================##

  # Calculate the midpoint velocity:

  v_mid = v + a * dt / 2

  # Calculate the midpoint position:

  x_mid = x + v * dt / 2

  # Update the local state:

  state_local['Position'] = x_mid

  state_local['Velocity'] = v_mid

  # Calculate the midpoint acceleration:

  a_mid = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the Updated Values:                                                             ##
  ##===========================================================================================##

  # Update the position:

  x += v_mid * dt

  # Update the velocity:

  v += a_mid * dt

  ##===========================================================================================##
  ## Update the State:                                                                         ##
  ##===========================================================================================##

  # Update the local state position:

  state_local['Position'] = x

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  ##===========================================================================================##
  ## Return the Updated State:                                                                 ##
  ##===========================================================================================##

  return state_local

##=============================================================================================##
## Define the 1D Runge-Kutta Change Function:                                                  ##
##=============================================================================================##

def cf_runge_kutta_1d(state, parameters, acceleration):

  # Create a local copy of the state object:

  state_local = state.copy()

  # Unpack the state:

  y = state_local['Position']

  v = state_local['Velocity']

  # Unpack the parameters:

  dt = parameters['dt']

  # Compute the acceleration:

  a = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the k1 Values (Slope at the Beginning of the Time Step):                        ##
  ##===========================================================================================##

  k1_y = v

  k1_v = a

  ##===========================================================================================##
  ## Calculate the k2 Values (Slope at the Midpoint of the Time Step, Based on k1 Values):     ##
  ##===========================================================================================##

  # Position k2 values:

  k2_y = v + 0.5 * k1_v * dt

  # Velocity k2 values:

  y_mid_1 = y + 0.5 * k1_y * dt

  v_mid_1 = v + 0.5 * k1_v * dt

  # Acceleration k2 values:

  state_local['Position'] = y_mid_1

  state_local['Velocity'] = v_mid_1

  k2_v = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the k3 Values (Slope at the Midpoint of the Time Step, Based on k2 Values):     ##
  ##===========================================================================================##

  # Position k3 values:

  k3_y = v + 0.5 * k2_v * dt

  # Velocity k3 values:

  y_mid_2 = y + 0.5 * k2_y * dt

  v_mid_2 = v + 0.5 * k2_v * dt

  # Acceleration k3 values:

  state_local['Position'] = y_mid_2

  state_local['Velocity'] = v_mid_2

  k3_v = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the k4 Values (Slope at the End of the Time Step, Based on k3 Values):          ##
  ##===========================================================================================##

  # Position k4 values:

  k4_y = v + k3_v * dt

  # Velocity k4 values:

  y_end = y + k3_y * dt

  v_end = v + k3_v * dt

  # Acceleration k4 values:

  state_local['Position'] = y_end

  state_local['Velocity'] = v_end

  k4_v = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Update the State Values:                                                                  ##
  ##===========================================================================================##

  # Update the object's velocity:

  v += (k1_v + 2.0 * k2_v + 2.0 * k3_v + k4_v) * dt / 6.0

  # Update the object's position:

  y += (k1_y + 2.0 * k2_y + 2.0 * k3_y + k4_y) * dt / 6.0

  # Update the local state positions:

  state_local['Position'] = y

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  # Return the updated local state:

  return state_local

##<font size = 5> <b> 2.5 Run and Analyze Simulation Functions </b> </font>

<br>

### <b> Run Simulation Function </b>

Next, we need a `run_simulation` function which will look very similar to what we've used before:

In [ ]:
##=============================================================================================##
## Create a Function to Set Up and Run the Simulation:                                         ##
##=============================================================================================##

def run_simulation(parameters, initial_state, change_function, acceleration, show_wtime = False):

  ##===========================================================================================##
  ## Get the Simulation Start Time:                                                            ##
  ##===========================================================================================##

  wall_start_time = tm.time()

  ##===========================================================================================##
  ## Unpack the Simulation Parameters:                                                         ##
  ##===========================================================================================##

  # Get the end time:

  end_time = parameters.loc['t_end']

  ##===========================================================================================##
  ## Create a Copy of the Initial State:                                                       ##
  ##===========================================================================================##

  state = initial_state.copy()

  ##===========================================================================================##
  ## Create the System's State History:                                                        ##
  ##===========================================================================================##

  # Create the state_history DataFrame:

  state_history = pd.DataFrame()

  state_history.name = 'Motion Over Time'

  state_history.index.name = 'Time Step'

  # Set the first entry in the state_history to be the initial system state:

  state_history[0] = state

  state_history = state_history.T

  ##===========================================================================================##
  ## Run the Simulation for the Desired Number of Time Steps:                                  ##
  ##===========================================================================================##

  end_condition = False

  while (end_condition == False):

    # Get the current time step:

    i = state_history.index[-1]

    # Get the state of the system for the current time step:

    state = change_function(state, parameters, acceleration)

    # Store the new system state in the state_history DataFrame:

    state_history.loc[i + 1] = state.T

    # If an end condition has been reached, break the loop:

    if (state['Time'] >= end_time): end_condition = True

    if (state['Position'] <= 0.0): end_condition = True

  ##===========================================================================================##
  ## Get the Simulation End Time:                                                              ##
  ##===========================================================================================##

  wall_end_time = tm.time()

  if (show_wtime == True):

    print("Elapsed Simulation Runtime:", np.round(wall_end_time - wall_start_time, 5), "seconds")

  ##===========================================================================================##
  ## Return the System's State History:                                                        ##
  ##===========================================================================================##

  return state_history

### <b>Computing the Loss With a Loss Function </b>

To compute the loss, we need to account for all of the different forms of energy that are going into our problem.

1) Potential Energy (gravity)

2) Potiential Energy (spring)

2) Kinetic Energy

3) Work (air drag)

In [ ]:
##=============================================================================================##
## Create a Function to Calculate Loss:                                                        ##
##=============================================================================================##

def loss_1d(parameters, state_history, acceleration):

  ##===========================================================================================##
  ## Calculate the Initial Total Energy:                                                       ##
  ##===========================================================================================##

  total_energy_initial = parameters['g'] * state_history.iloc[0]['Position']

  ##===========================================================================================##
  ## Calculate the Total Energy History:                                                       ##
  ##===========================================================================================##

  total_energy_history = parameters['g'] * state_history.iloc[:]['Position'] + \
                         0.5 * state_history.iloc[:]['Velocity'] ** 2

  ##===========================================================================================##
  ## Calculate the Energy Lost to Non-Conservative Forces (Work is force times velocity * dt): ##
  ##===========================================================================================##

  v  = state_history.iloc[:]['Velocity']
  y  = state_history.iloc[:]['Position']
  dt = parameters['dt']
  mass = parameters['mass']
  y_attach = parameters['y_attach']
  L = parameters["length"]
  k = parameters['k']

  rho = parameters['rho']
  area = parameters['area']
  C_d = parameters['Cd']

  distance_to_attach = np.abs(y_attach - y)

  extension = distance_to_attach - L

  extension = np.maximum(extension, 0)

  acc = acceleration(state_history, parameters) + parameters['g'] - k * extension / mass

  work = (v * acc * dt).cumsum()

  e_bungee = 0.5 * k * extension**2 / mass

  ##===========================================================================================##
  ## Calculate the Loss History:                                                               ##
  ##===========================================================================================##

  loss_history = (total_energy_history + e_bungee - total_energy_initial - work) / \
                 total_energy_initial

  ##===========================================================================================##
  ## Return the Loss History:                                                                  ##
  ##===========================================================================================##

  return loss_history

### <b> Analyze Simulation Function </b>

Finally, we need an `analyze_simulation` function which will look very similar to what we've used before:

In [ ]:
##=============================================================================================##
## Create a Function to Analyze the Simulation Results:                                        ##
##=============================================================================================##

def analyze_simulation(state_history, parameters, loss_function, acceleration, x_list = [],
                       y_list = [], color_list = [], title_list = [], x_label_list = [],
                       y_label_list = [], aspect_list = [], plot_data = False, display_data = False):

  ##===========================================================================================##
  ## Compute the Simulation_Loss:                                                              ##
  ##===========================================================================================##

  loss_history = loss_function(parameters, state_history, acceleration)

  ##===========================================================================================##
  ## Add the Loss History to the State History:                                                ##
  ##===========================================================================================##

  state_history['Loss'] = loss_history

  ##===========================================================================================##
  ## Calculate the Total Loss:                                                                 ##
  ##===========================================================================================##

  total_loss = loss_history.sum()

  ##===========================================================================================##
  ## If requested, display the history data:                                                   ##
  ##===========================================================================================##

  if (display_data == True):

    # View the system's state history and final state:

    display_dataframes([state_history, state_history.iloc[-1]],["State History", "Final State"],
                       n_items = 7)

  ##===========================================================================================##
  ## If requested, plot the history data:                                                      ##
  ##===========================================================================================##

  if (plot_data == True):

    # Get the settings for each graph to be made:

    for X, Y, color, title, x_label, y_label, aspect in \
      zip(x_list, y_list, color_list, title_list, x_label_list, y_label_list, aspect_list):

      # Create the graph:

      x_size = 10
      y_size = 10 * aspect

      ax = state_history.plot(
        kind    = 'line',
        y       = [Y],
        x       = str(X),
        color   = [color],
        figsize = (x_size, y_size)
      )

      # Set the graph title:

      ax.set_title(title, fontsize = 14)

      # Set the Graph x-axis label:

      ax.set_xlabel(x_label, fontsize = 12)

      # Set the Graph y-axis label:

      ax.set_ylabel(y_label, fontsize = 12)

      # Add grid and suppress the legend:

      ax.grid(True, linestyle = '--', alpha = 0.5)

      legend = ax.legend()

      legend.remove()

    # Display the plots:

    plt.show()

  ##===========================================================================================##
  ## Return the History Data:                                                                  ##
  ##===========================================================================================##

  return state_history.iloc[-1]['Loss']

##<font size = 5> <b> 2.6 Running the Simulations </b> </font>

<br>

### <b> Simulation Without Drag Force </b>

For our first run, let's test the simulation using the accelration without drag and see if our results make sense.

In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Set the desired parameter set:

parameters = parameters_bungee.copy()

# Set the simulation end time:

parameters['t_end'] = 30

parameters['dt'] = 0.01

# Set the desired acceleration function:

acceleration = acceleration_bungee_1d

# Set the desired change function:

change_function = cf_runge_kutta_1d

# Run the simulation:

history_no_drag = run_simulation(parameters, initial_state, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_no_drag, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)

We see that the bungee-jumper's position oscillates sinusoidally with time. This is what we expect to happen in the absence of any damping forces (such as air drag).

<br>

### <b> Simulation With Drag Force </b>

Now, let's use the acceleration function that includes the air drag and see how our simulation is affected.

In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Set the desired parameter set:

parameters = parameters_bungee.copy()

# Set the simulation end time:

parameters['t_end'] = 30

parameters['dt'] = 0.01

# Set the desired acceleration function:

acceleration = acceleration_bungee_drag_1d

# Set the desired change function:

change_function = cf_runge_kutta_1d

# Run the simulation:

history_drag = run_simulation(parameters, initial_state, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_drag, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)

Again, we see a sinusoidal motion of the bungee-jumper, just as before, only now, the amplitude of the oscillations diminishes with time. This is happening because the work due to air drag is extracting energy from the system proportionally to the speed of the jumper. As such the oscillations will die out over time.

<br>

This type of motion is called an 'under-damped' oscillator.

<br>

### <b> Simulation With Artificially High Drag Force </b>

Next, let's examine what happens to the motion of the bungee jumper if we play around with the coefficient of air drag (Cd).

<br>

Let's artificially increase the value of Cd to a huge number (200) and see what happens to the bungee-jumper's motion.

In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Set the desired parameter set:

parameters = parameters_bungee.copy()

# Set the simulation end time:

parameters['t_end'] = 30

parameters['dt'] = 0.01

parameters['Cd'] = 200

# Set the desired acceleration function:

acceleration = acceleration_bungee_drag_1d

# Set the desired change function:

change_function = cf_runge_kutta_1d

# Run the simulation:

history_extra_drag = run_simulation(parameters, initial_state, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_extra_drag, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)

Now, se see that the bungee jumper doesn't even really undergo any oscillations, this is because the drag work is extracting so much energy that there isn't really any left for them to bounce up and down with.

<br>

An oscillating system in this situation is called "over-damped".

### <b> Viewing the Results Together </b>

Let's overlay our three simulations and look at them simultaneously.

In [ ]:
##=============================================================================================##
## Examine Height Vs. Time:                                                                    ##
##=============================================================================================##

data_list = [history_no_drag, history_drag, history_extra_drag]
y_list   = ['Position']
x_list   = ['Time']
legend   = ['No Drag', 'Drag', 'Extra Drag']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time']
y_labels = ['Height (m)']
x_labels = ['Time (s)']
aspects  = [0.6]

plot_data(data_list, x_list, y_list, colors, titles, x_labels, y_labels, aspects, legend,
          overplot = True)

In [ ]:
##=============================================================================================##
## Examine Velocity Vs. Time:                                                                  ##
##=============================================================================================##

data_list = [history_no_drag, history_drag, history_extra_drag]
y_list   = ['Velocity']
x_list   = ['Time']
legend   = ['No Drag', 'Drag', 'Extra Drag']
colors   = ['blue', 'red', 'green']
titles   = ['Velocity Vs Time',]
y_labels = ['Height (m)']
x_labels = ['Time (s)']
aspects  = [0.6]

plot_data(data_list, x_list, y_list, colors, titles, x_labels, y_labels, aspects, legend,
          overplot = True)

##<font size = 5> <b> 2.7 Solving for the Ideal Bungee Cord Length </b> </font>

Jumping back to our main goal for this simulation, let's try and find the ideal length for the bungee-cord rope.

<br>

We could find the proper $L$ by the "guess-and-check" method, but we have tools that are more efficient and accurate than that.  Let's use `root_scalar()` to find the optimal length $L$.

<br>

### <b> Using Root Scalar </b>

Remember that  `root_scalar()` finds the roots of a mathematical function.  A function always has an independent variable and a dependent variable.   In this case our function,  $\Delta y = f(L)$, has $L$ as the independent variable and $\Delta y$ as the dependent variable, where $\Delta y$ is the distance between the jumper's lowest point and the tea cup (that is, the "error" between the result and the ideal results).

<br>  

We just need to put this mathematical function in the form of a Python function:

In [ ]:
##=============================================================================================##
## Define an Error Function for the Bungee Cord Length:                                        ##
##=============================================================================================##

def error_function(L, y_ideal, parameters, initial_state, acceleration, change_function):

  # Set the desired parameter set:

  parameters = parameters.copy()

  # Set the rope length for the current simulation:

  parameters['length'] = L

  # Set the desired acceleration function:

  acceleration = acceleration_bungee_drag_1d

  # Set the desired change function:

  change_function = cf_runge_kutta_1d

  # Run the simulation:

  history_fall = run_simulation(parameters, initial_state, change_function, acceleration)

  # Get the minimum vertical position during the jump:

  y_min = min(history_fall['Position'])

  # Compute the difference between the ideal and actual values:

  delta_y = y_min - y_ideal

  # Return the difference:

  return delta_y

`error_func` takes in the independent variable as an argument, and spits out the dependent variable. Let's test the error function with the value of $L$ that we used above. We should get a minimum value of about 4.38 m.

In [ ]:
##=============================================================================================##
## Test the Error Function for the Bungee Cord Length:                                         ##
##=============================================================================================##

# Set the value of the bungee-cord to test:

L_test = 25.0 # meters

# Set the 'ideal' value for the minimum vertical position:

y_ideal = 0.0 # meters

# Run the error function and get the result:

error = error_function(L_test, y_ideal, parameters_bungee, initial_state,
                       acceleration_bungee_drag_1d, cf_euler_cromer_1d)

# Print out the result:

print()
print("The error for L = " + str(np.round(L_test,2)) + " m is " + str(np.round(error,2)) + " m.")

Now we can pass our error function into `root_scalar`, and it should find the value of $L$ that will optimize the length of our cord.   Since `root_scalar` needs a 'bracket' that gives a negative and a positive result, we'll give it a two lengths for $L$, one that we know is too short and one that is too long:

In [ ]:
##=============================================================================================##
## Use the Root Scalar Function to Find the Ideal Bungee Cord Length:                          ##
##=============================================================================================##

L_ideal = spo.root_scalar(error_function, args=(y_ideal, parameters_bungee, initial_state,
                          acceleration_bungee_drag_1d, cf_euler_cromer_1d), bracket=[25, 80])

print("The ideal length for the bungee cord is " + str(np.round(L_ideal.root, 2)) + " m.")

And now we can test and see that at 28.1 meters, our cord has the ideal length to dip our biscuit into a cup of piping hot British tea.

In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Set the desired parameter set:

parameters = parameters_bungee.copy()

# Set the bungee-cord length:

parameters['length'] = L_ideal.root

# Set the desired acceleration function:

acceleration = acceleration_bungee_drag_1d

# Set the desired change function:

change_function = cf_runge_kutta_1d

# Run the simulation:

history_fall = run_simulation(parameters, initial_state, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_fall, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)

[Return to Top](#Notebook-Start)

<a name="RopeMass"></a>

---

#<font size = 6> <b> 3. Adding Rope Mass </b> </font>

---

##<font size = 5> <b> 3.1 Introducing the Problem </b> </font>

In the previous section, we simulated a bungee jump with a model that took into account gravity, air resistance, and the spring force of the bungee cord, but we ignored the weight of the cord.  Now we'll include that factor, using the paper by [Heck, Uylings, and Kędzierska](http://iopscience.iop.org/article/10.1088/0031-9120/45/1/007).

<br>

Just a word about the counter-intuitive nature of this process.  As Heck et al. argue, the increased acceleration created by the chain is a result of the more general form of the Second Law:

<br>

$$\Sigma F = \frac{dp_{obj}}{dt} = \frac{dm_{obj}}{dt} v_{obj} +m_{obj}a_{obj}$$

<br>

where $p$ is momentum (i.e. $mv$).  If you have taken calculus, you might recognize the use of the product rule here (if you haven't, no big deal!).  But what does this mean?  And how is it possible that acceleration is *not* equal to $\frac{F}{m}$?!

<br>

Consider a military space ship cruising through space with its engines off.  There is no outside force being applied to the system, so it is not accelerating, but it is moving at a quick constant velocity.  It is dropping a space mine off every couple of seconds, but it doesn't want the mine to move after its been dropped off.  In essence, the ship has to *push* the mine backwards to reduce the mine's velocity to zero, and in doing so the mine pushes the spaceship forward (this is a Third Law, equal/opposite reaction).  So by jettisoning mass and slowing that mass down, the ship is sped up.  The larger the mine, and the faster the ship, the larger the effect! This is like the falling bungee jumper leaving sections of rope behind: the momentum that the rope loses as it stops is passed onto the remaining rope and the object.  

<br>

That's some crazy physics for you, right there.  Yessirree!

<br>

##<font size = 5> <b> 3.2 Setting up the Problem </b> </font>


We'll make the same assumptions we did in the previous notebook, but add this new force.  Here's is a `params` dictionary that holds the parameters, including a cord mass of 75 kg:


In [ ]:
##=============================================================================================##
## Set the Simulation Parameters:                                                              ##
##=============================================================================================##

# Local acceleration due to gravity (in m/s^2):

g = 9.8

# Terminal velocity (in m/s):

v_term = 70

# Air mass density (in kg/m^3):

rho = 1.2

# Object mass (in kg):

mass_person = 75

mass_cord = 75

# Object's cross-sectional area (in m^2):

area = 0.3

# Calculate the object's drag coeffcient:

C_d = (2.0 * mass * g) / (rho * v_term**2 * area)

print("The object's drag coefficient is: " + str(np.round(C_d, 3)))

# Bungee cord's attachment location (in m):

y_attach = 80

# Bungee cord's unstretched length (in m):

length = 25.0

# Bungee cord's spring constant (in N / m):

k = 40.0

# Time step size (in seconds):

dt = 0.01

# Simulation end time (in seconds):

t_end = 30

##=============================================================================================##
## Store the parameters in a Pandas Series:                                                    ##
##=============================================================================================##

parameters_bungee_mass = pd.Series({
    "g":  g,
    "Cd": C_d,
    "rho": rho,
    "mass_person": mass_person,
    "mass_cord": mass_cord,
    "area": area,
    "y_attach": y_attach,
    "length": length,
    "k": k,
    "dt": dt,
    "t_end": t_end
})

# Rename the columns:

parameters_bungee_mass.name = 'Value'

parameters_bungee_mass.index.name = "Parameter"

##=============================================================================================##
## Create the State Object                                                                     ##
##=============================================================================================##

# Initial Height:

y_initial = 80.0 # meters

# Initial Velocity:

v_initial = 0 # meters per second

# Initial Time:

t_initial = 0 # seconds

##=============================================================================================##
## Store the initial state in a Pandas Series:                                                 ##
##=============================================================================================##

initial_state = pd.Series({
    "Position" : y_initial,
    "Velocity" : v_initial,
    "Time"     : t_initial
})

initial_state.name = 'Value'

initial_state.index.name = "State"

##=============================================================================================##
## Display the Parameters and Initial State:                                                   ##
##=============================================================================================##

display_dataframes([parameters_bungee_mass, initial_state],
                   ["Bungee Jump Parameters", "Initial State"], n_items = 15)

##<font size = 5> <b> 3.3 Adding the New Acceleration Term </b> </font>

Now we need to add the new force. Here is the equation for the contribution to acceleration made by the cord:

$a_{cord} = \frac{\frac{1}{2}\mu v^2}{\mu(L-y) + 2L}$

Because the cord creates an acceleration in two ways (its mass and its stretching), you are going to need some kind of `if` clause to determine which calculation should be used.

<br>

Here mu is the ratio of the cord mass to the person's mass: $\mu = \dsfrac{m_{\text(cord)}}{m_{\text(person)}}$